# DEM 实际高程剖面提取

**用途：** 从 DEM 中选择矩形区域，沿横向、纵向或任意两点提取一段真实高程。
默认按源文件声明的米制平面坐标计距，不猜测 EPSG，不减去海拔均值。

流程：**安装依赖 → 修改路径 → 查看 DEM → 选择范围与方向 → 更新剖面 → 导出**。
最后导出的 `surface_profile.csv` 仅两列 `x_m,elevation_m`，可供二维波场程序直接导入。

同目录需保留 `dem_profiles.py`。当前 DEM 是 GDAL 不能直接识别的投影代码 9 固定记录变体，
模块已经加入校验读取分支，原始文件保持只读。

## 1. 首次安装依赖

选择 Python 3.10+ 内核。若提示缺少模块，取消下一单元安装命令的注释后运行一次，再重启内核。
如已有依赖，可以直接继续。VS Code 和 JupyterLab 均可使用；鼠标框选额外需要 ipympl 前端支持。

In [ ]:
# 首次使用时，取消注释；安装在当前 Notebook 内核对应的 Python 环境。
# %pip install --index-url https://pypi.org/simple "numpy>=1.24" "pandas>=2" "matplotlib>=3.7,<4" "rasterio>=1.4.4,<2" "pyproj>=3.6,<4" "ipywidgets>=8,<9" "ipympl>=0.9,<1"

In [ ]:
from pathlib import Path
import os, sys, json

# 兼容 VS Code 从工作区目录、或 Jupyter 从 Notebook 所在目录启动。
module_dir = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidate in (base, base/'tools/dem_profiles', base/'Seis3D_Curvilinear/tools/dem_profiles'):
        if (candidate/'dem_profiles.py').is_file():
            module_dir = candidate.resolve()
            break
    if module_dir is not None:
        break
if module_dir is None:
    raise FileNotFoundError('请把 Notebook 和 dem_profiles.py 放在同一目录，并从该目录启动。')
sys.path.insert(0, str(module_dir))
from dem_profiles import (DEM, project_output, line_in_roi, extract_profile,
                          longest_valid_segment, plot_map, plot_profile, export_profile)

OUTPUT_DIR = project_output()       # 项目内运行时，自动使用外层 out/dem_profiles
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR/'cache/matplotlib'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
%matplotlib inline

## 2. 输入路径与坐标约定

修改 `DEM_PATH` 即可换文件。对当前文件，无需填写坐标系和单位覆盖参数。
`grid` 表示源文件米制投影平面中的水平距离；只有以后确认 CRS 后，才考虑 `geodesic`。
未知单位会报错，不会自动把角度当米。

In [ ]:
DEM_PATH = Path(r'D:/Data file/航飞ECW/导出区域数据/3_jingbiancenter.dem')
CRS_OVERRIDE = None             # 不猜测 EPSG；仅在确认后填写，例如 'EPSG:xxxx'
XY_UNIT_OVERRIDE = None         # 当前文件声明水平单位为米，保持 None
Z_UNIT_OVERRIDE = None          # 当前文件声明垂直单位为米，保持 None
DISTANCE_MODE = 'grid'          # 默认米制格网距离，不作经纬度或垂直基准转换

if 'dem' in globals():
    dem.close()
dem = DEM(DEM_PATH, output_dir=OUTPUT_DIR, crs_override=CRS_OVERRIDE,
          xy_unit_override=XY_UNIT_OVERRIDE, z_unit_override=Z_UNIT_OVERRIDE)
info = dem.info()
display(pd.DataFrame({
    '项目': ['源文件', '读取方式', '行×列', '像元间距（源单位）', '高程单位', '水平每单位米数', '确认的 CRS', '输出目录'],
    '值': [str(DEM_PATH), '校验固定记录兼容分支' if info['fallback_reader'] else 'Rasterio/GDAL',
           str(info['shape']), str(info['resolution']), str(info['height_unit']),
           str(info['horizontal_metres_per_unit']), str(info['crs']), str(OUTPUT_DIR)]
}))
print('注意：更密的输出采样属于插值，不会增加原始 DEM 的实际地形分辨率。')

## 3. 地形总览

坐标轴保持原 DEM 的 X/Y。颜色显示实际海拔（米），不是归一化高度。
大栅格的地图预览会降采样，剖面采样仍读取原分辨率数据。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), layout='constrained')
image = plot_map(dem, ax)
fig.colorbar(image, ax=ax, label='Elevation (m)')
ax.set_title(DEM_PATH.name)
plt.show()
plt.close(fig)

## 4. 选择区域、方向和采样范围

- **横向：** 固定 Y，沿 X 从小到大。位置 0 是矩形下侧，1 是上侧。
- **纵向：** 固定 X，沿 Y 从小到大。位置 0 是矩形左侧，1 是右侧。
- **任意两点：** 使用输入的起终点，两个点均需落在所选矩形内。
- **起止距离：** 相对上面定义的整条线，单位米；终点 0 表示到末尾。
- **采样间距：** 建议先取 10–40 m；可取更小间距，但它仍是约 38.2 m 源像元的插值。

默认矩形位于 DEM 中央，米制数据使用约 1000×1000 m 的区域。可以拖动滑块，也可以直接修改两端数值。
修改参数后点击“更新剖面”；导出时会重新应用当前控件值。

In [ ]:
left, bottom, right, top = dem.bounds
cx, cy = (left+right)/2, (bottom+top)/2
half_x = min((right-left)*.4, 500/dem.xy_factor) if dem.xy_factor else (right-left)*.1
half_y = min((top-bottom)*.4, 500/dem.xy_factor) if dem.xy_factor else (top-bottom)*.1
step = min(dem.source.res)/10
wide = widgets.Layout(width='98%')
style = {'description_width': '140px'}
roi_x = widgets.FloatRangeSlider(value=(cx-half_x,cx+half_x), min=left, max=right, step=step,
                                description='X 范围', readout_format='.2f', continuous_update=False, layout=wide, style=style)
roi_y = widgets.FloatRangeSlider(value=(cy-half_y,cy+half_y), min=bottom, max=top, step=step,
                                description='Y 范围', readout_format='.2f', continuous_update=False, layout=wide, style=style)
direction = widgets.Dropdown(options=[('横向 / X 增大','horizontal'),('纵向 / Y 增大','vertical'),('任意两点','custom')],
                             value='horizontal', description='方向', style=style)
position = widgets.FloatSlider(value=.5,min=0,max=1,step=.01,description='剖面位置 0–1',continuous_update=False,style=style,layout=wide)
spacing = widgets.BoundedFloatText(value=10,min=.0001,max=100000,description='输出间距（m）',style=style)
start_dist = widgets.BoundedFloatText(value=0,min=0,max=1e9,description='起始距离（m）',style=style)
end_dist = widgets.BoundedFloatText(value=0,min=0,max=1e9,description='终点（0=全长）',style=style)
interpolation = widgets.Dropdown(options=[('双线性','bilinear'),('最近邻原像元','nearest')],value='bilinear',description='插值',style=style)
trim_invalid = widgets.Checkbox(value=False,description='只保留最长连续有效段（会缩短选取范围）',layout=wide)
x0 = widgets.FloatText(value=cx-half_x,description='起点 X',style=style)
y0 = widgets.FloatText(value=cy-half_y,description='起点 Y',style=style)
x1 = widgets.FloatText(value=cx+half_x,description='终点 X',style=style)
y1 = widgets.FloatText(value=cy+half_y,description='终点 Y',style=style)
custom_box = widgets.VBox([widgets.HBox([x0,y0]),widgets.HBox([x1,y1])])
refresh_button = widgets.Button(description='更新剖面',button_style='info')
export_button = widgets.Button(description='导出当前参数',button_style='success')
result_output, export_output = widgets.Output(), widgets.Output()
state = {}

def current_profile():
    roi = (*roi_x.value,*roi_y.value)
    p0,p1 = line_in_roi(roi,direction.value,position.value,[(x0.value,y0.value),(x1.value,y1.value)])
    table,metadata = extract_profile(dem,p0,p1,spacing.value,start_dist.value,
                                     None if end_dist.value==0 else end_dist.value,
                                     interpolation.value,DISTANCE_MODE)
    if trim_invalid.value:
        table,metadata = longest_valid_segment(table,metadata)
    return table,metadata,roi

def refresh_profile(_=None):
    with result_output:
        clear_output(wait=True)
        try:
            table,metadata,roi = current_profile()
            state.update(table=table,metadata=metadata,roi=roi)
            print(f'完整线长 {metadata["full_line_length_m"]:.3f} m；当前输出 {table.x_m.iloc[-1]:.3f} m；'
                  f'{len(table)} 个样点；无效点 {metadata["invalid_samples"]}')
            if metadata.get('longest_valid_segment_applied'):
                print('已明确裁剪到最长有效段：',metadata['kept_source_distance_range_m'],
                      'm；丢弃样点',metadata['discarded_samples'])
            fig,axes = plt.subplots(1,2,figsize=(13,4.2),layout='constrained')
            image = plot_map(dem,axes[0],table,roi)
            axes[0].set_xlim(roi[0],roi[1]); axes[0].set_ylim(roi[2],roi[3])
            axes[0].set_title('Selected ROI / profile location')
            fig.colorbar(image,ax=axes[0],label='Elevation (m)')
            plot_profile(table,axes[1]); axes[1].set_title('Actual elevation profile')
            plt.show(); plt.close(fig)
            display(table.head())
        except Exception as exc:
            state.clear()
            print(type(exc).__name__+': '+str(exc))

def save_current_profile(_=None):
    with export_output:
        clear_output(wait=True)
        try:
            # Always rebuild from the current controls, not stale state from a previous plot.
            table,metadata,roi = current_profile()
            folder = export_profile(dem,table,metadata,OUTPUT_DIR,roi)
            state.update(table=table,metadata=metadata,roi=roi,export_folder=folder)
            print('已导出：',folder)
            print('模拟输入：',folder/'surface_profile.csv')
            print('原始 DEM 坐标与选择参数分别保存在 profile_with_coordinates.csv 和 selection.json。')
        except Exception as exc:
            print(type(exc).__name__+': '+str(exc))

refresh_button.on_click(refresh_profile)
export_button.on_click(save_current_profile)
display(widgets.VBox([roi_x,roi_y,widgets.HBox([direction,interpolation]),position,
                      widgets.HBox([spacing,start_dist,end_dist]),custom_box,trim_invalid,
                      widgets.HBox([refresh_button,export_button]),result_output,export_output]))
refresh_profile()

## 5. 可选：在 DEM 图上鼠标框选

上面的滑块和数值输入不依赖交互绘图后端，默认即可使用。
若需要鼠标框选，将下方 `ENABLE_MOUSE_SELECTION=True` 后运行；在图上按住鼠标左键拖矩形，
范围会写回上方 X/Y 控件，再点击“更新剖面”。保存对 `selector` 的引用是为了避免选择器被回收。
若前端不支持 widget 后端，请继续使用数值控件。

In [ ]:
ENABLE_MOUSE_SELECTION = False
if ENABLE_MOUSE_SELECTION:
    get_ipython().run_line_magic('matplotlib','widget')
    from matplotlib.widgets import RectangleSelector
    mouse_figure,mouse_axis = plt.subplots(figsize=(10,6),layout='constrained')
    image = plot_map(dem,mouse_axis)
    mouse_figure.colorbar(image,ax=mouse_axis,label='Elevation (m)')
    mouse_axis.set_title('Drag with the left mouse button to select the ROI')
    def on_rectangle(start,end):
        if None in (start.xdata,start.ydata,end.xdata,end.ydata):
            return
        xa,xb = sorted([max(left,min(right,start.xdata)),max(left,min(right,end.xdata))])
        ya,yb = sorted([max(bottom,min(top,start.ydata)),max(bottom,min(top,end.ydata))])
        if xa < xb and ya < yb:
            roi_x.value=(xa,xb)
            roi_y.value=(ya,yb)
    selector = RectangleSelector(mouse_axis,on_rectangle,useblit=False,button=[1],interactive=True)
    plt.show()
else:
    print('鼠标框选未启用；上方范围滑块和数值输入可以直接使用。')

## 6. 可选：纯代码精确截取

不使用控件时可直接写端点或 ROI。例如把下面的开关改为 True，可沿当前 ROI 的纵向中线，
从 100 m 截至 900 m，以 10 m 间距输出一条 800 m 剖面。前提是完整线长至少 900 m。
NoData 不会被填成零，默认拒绝导出；需要时请明确调用 `longest_valid_segment`。

In [ ]:
RUN_MANUAL_EXAMPLE = False
if RUN_MANUAL_EXAMPLE:
    roi = (*roi_x.value,*roi_y.value)
    p0,p1 = line_in_roi(roi,direction='vertical',position=.5)
    manual_table,manual_metadata = extract_profile(dem,p0,p1,spacing_m=10,
                                                   start_m=100,end_m=900,
                                                   method='bilinear',distance_mode='grid')
    manual_folder = export_profile(dem,manual_table,manual_metadata,OUTPUT_DIR,roi)
    display(manual_table.head())
    print(manual_folder)

## 7. 可选：批量提取横向和纵向测线

将下面的开关改为 True，会在当前矩形的 25%、50%、75% 位置各提取横向和纵向剖面，
分别保存到独立文件夹，坐标和参数随数据保存。

In [ ]:
RUN_BATCH = False
if RUN_BATCH:
    roi = (*roi_x.value,*roi_y.value)
    for mode in ('horizontal','vertical'):
        for fraction in (.25,.5,.75):
            p0,p1 = line_in_roi(roi,mode,fraction)
            table,metadata = extract_profile(dem,p0,p1,spacing_m=spacing.value,
                                               method=interpolation.value,distance_mode=DISTANCE_MODE)
            metadata.update(batch_direction=mode,batch_position=fraction)
            print(mode,fraction,export_profile(dem,table,metadata,OUTPUT_DIR,roi))

## 8. 导入二维波场程序

1. 在“外部地表高程”中选择导出的 `surface_profile.csv`，载入后检查覆盖范围。
2. **实际海拔数据保留原值。** 通常保留导入器自动计算的高程均值，作为模型局部坐标基准；
   不要把人工零高程示例的基准 0 m 直接套到约 1000 多米的实际海拔上。
3. 需满足 `(Nx−1)h ≤ 剖面长度`。恰好 1000 m 时，可从 Nx=401、h=2.5 m 开始设置，
   Nz、地形展平深度、主频和物性另外检查。DEM 输出间距与波场网格间距是不同参数。
4. “完成”后“重置并应用参数”，核对生效网格与 dt。

输出保留实际高程，无归一化、无去趋势、无自动平滑。它反映源 DEM，并不表示已验证源测绘数据的绝对精度。
操作结束后可手动运行 `dem.close()` 释放文件；操作控件期间请保持数据集打开。

更多格式说明见同目录 README；技术接口依据：
[Rasterio](https://rasterio.readthedocs.io/en/stable/topics/reading.html)、
[GDAL USGSDEM](https://gdal.org/en/stable/drivers/raster/usgsdem.html)、
[PyProj](https://pyproj4.github.io/pyproj/stable/api/crs/crs.html)、
[Matplotlib RectangleSelector](https://matplotlib.org/stable/gallery/widgets/rectangle_selector.html)。